# 跨靶点臂结构分析 —— 36 条轨迹

> **本 notebook 由 `make_cross_notebooks.py` 从主臂 V3 版定点替换生成，非重写。**
> 与主臂版的全部差异即该脚本的替换表，每一处均断言命中次数。
> **白名单 `ALLOWED_ARMS` 已刻意加入 `cross_solv_v3`** —— 原白名单会正确地拒绝本臂；
> 扩充它是 2026-10-19 的一次决定，须随本 notebook 一同入档。
> 主臂产物（`step7_solv_v3` / `step6e_solv_v3` / `*_SOLV_V3.csv`）不受本notebook 影响。

---

# phase4 · 结构分析（跨靶点臂 cross_solv_v3）

在 `md_fit.xtc` 上算 COMPASS 的四个结构轴：
`rmsd_prot_mean` · `rmsd_lig_mean` · `mindist_mean`(nm) · `hbond_occupancy`。
取**后 50 ns**、`STRIDE = 10`，与 V2 协议一致。

**相对 V2 的改动**

| | V2 | V3 |
|---|---|---|
| `ARMS` | `md_solv`, `md_cross_solv` | `cross_solv_v3` |
| 配体选择 | `LIG = 'UNL'`，脚本内三处 `resname UNL` 写死 | 由 `index.ndx` **逐轨迹探测**，作参数传入 |
| 输出根 | `results/step6e_solv/` | `results/step6e_cross_v3/`（不覆盖 V2） |
| 产物字段 | — | 新增 `ligand_resname`，使 JSON 自带来源 |

**窗口无须改动**：`start = n_total // 2` = 5000、STRIDE 10 → 501 帧 = 50.00–99.99 ns。

> Methods 需一次说清：MM-PBSA 用 200 帧（5001–10000，间隔 25，50.01–99.76 ns），
> 结构分析用 501 帧（5000 起，50.00–99.99 ns）。同为后 50 ns，但**不是同一取样**：501 帧（stride 10，自 5000 起）对 200 帧（interval 25，自 5001 起），且 5001 不落在 stride-10 的格点上，故两者互不包含。

In [7]:
# === 版本闸：确认这是 V3 notebook，且不会误触 V2 ===
# Num 1
from pathlib import Path
import sys

EXPECTED_ARM = "cross_solv_v3"

print("=" * 62)
print("  这是 V3 版 notebook —— 只处理 md_solv_v3，输出全部带 _v3 / _V3")
print("  V2 版（md_solv / md_cross_solv）是另一个文件，不要在同一会话中打开")
print("=" * 62)

_root = Path("/mnt/hpc/home/wanghongmei/lang2")
assert (_root / EXPECTED_ARM).is_dir(), f"缺 {EXPECTED_ARM}"
_n = len(list((_root / EXPECTED_ARM).glob("*/*/06_md/md_fit.xtc")))
print(f"{EXPECTED_ARM}: {_n} 条 md_fit.xtc （应为 36）")
assert _n == 36, f"md_fit.xtc 数量 {_n} != 36 —— phase2 跨臂是否已完成？"   # 36 的依据：AMD-2 2026-10-31

  这是 V3 版 notebook —— 只处理 md_solv_v3，输出全部带 _v3 / _V3
  V2 版（md_solv / md_cross_solv）是另一个文件，不要在同一会话中打开
cross_solv_v3: 36 条 md_fit.xtc （应为 36）


In [8]:
# === 配置 ===
# Num 2
from pathlib import Path
import os, glob, json
import pandas as pd, numpy as np

ROOT = Path("/mnt/hpc/home/wanghongmei/lang2")
ARMS = ["cross_solv_v3"]
N_EXPECT = 36   # 依据 AMD-2 2026-10-31：该臂以两批达到 36（30 + 6）
OUT_ROOT = ROOT / "results" / "step6e_cross_v3"    # 新输出根，不覆盖主臂
TRAJ   = "md_fit.xtc"
STRIDE = 10
SBATCH = dict(partition="gpu", account="hongmeilab", qos="normal",
              cpus=4, time="02:00:00", gres="")
CONCURRENCY = 20

ALLOWED_ARMS = {"md_solv", "md_cross_solv", "md_solv_v3", "md_cross_solv_v3",
                "cross_solv_v3"}        # ← 跨靶点臂目录，2026-10-19 刻意加入
assert set(ARMS) <= ALLOWED_ARMS and not (set(ARMS) & {"md", "md_cross"})
print("配置 OK | 输出根:", OUT_ROOT)

配置 OK | 输出根: /mnt/hpc/home/wanghongmei/lang2/results/step6e_cross_v3


In [9]:
# === 配体残基名：逐轨迹探测（V2=UNL / V3=MOL），并与 Node 05 的 I2b 对账 ===
# Num 3
import re, glob
def ndx_groups(ndx):
    return [m.group(1) for m in
            (re.match(r'\s*\[\s*(.+?)\s*\]', ln) for ln in open(ndx)) if m]

def lig_group(ndx, candidates=("MOL", "UNL")):
    g = ndx_groups(ndx)
    hit = [c for c in candidates if c in g]
    if len(hit) != 1:
        raise AssertionError(f"{ndx}: 配体组不唯一或缺失，命中 {hit}；现有组 {g}")
    return hit[0], g.index(hit[0])

ndxs = sorted(glob.glob(f"{ROOT}/{ARMS[0]}/*/*/06_md/index.ndx"))
assert ndxs, f"未找到 index.ndx —— {ARMS[0]} 是否已产出轨迹？"
names = {lig_group(n)[0] for n in ndxs}
assert len(names) == 1, f"配体残基名在轨迹间不一致: {names}"
LIG = names.pop()
print(f"✓ 配体残基名 = {LIG}  ({len(ndxs)} 条轨迹一致)")

# 配体原子数核验 —— **不与冻结记录对账**（理由同 phase3）。
# 实测 2026-10-19：Node05_prep_freeze_cross.json 无 invariants 块，
# Node05_prep_verify_cross.json 每体系只存 {all_passed, system, target}。
n_lig, meta = {}, {}
for n in ndxs:
    sysname = Path(n).parents[2].name            # pool_mol_XXX__REC__pose
    n_lig[sysname] = len(open(n).read().split(f"[ {LIG} ]")[1]
                         .split("[")[0].split())
    parts = sysname.split("__")
    meta[sysname] = (parts[0], parts[1])

# ① 同一分子在两受体上的配体原子数必须相同 —— I2b 的可计算形式
per_mol = {}
for sysname, cnt in n_lig.items():
    mol, rec = meta[sysname]
    per_mol.setdefault(mol, {})[rec] = cnt
bad = {m: d for m, d in per_mol.items() if len(set(d.values())) != 1}
assert not bad, f"同一分子在两受体上配体原子数不一致: {bad}"

# ② 逐分子打印，供与 §B12.3 人工对照；刻意不作断言（写死数字即 D14 陷阱）
print(f"✓ {len(n_lig)} 个体系；每分子在两受体上的配体原子数一致：")
for m in sorted(per_mol):
    print(f"    {m}: {sorted(per_mol[m].items())}")

✓ 配体残基名 = MOL  (36 条轨迹一致)
✓ 12 个体系；每分子在两受体上的配体原子数一致：
    pool_mol_001: [('3V3M', 55), ('4YLU', 55)]
    pool_mol_004: [('3V3M', 51), ('4YLU', 51)]
    pool_mol_012: [('3V3M', 51), ('4YLU', 51)]
    pool_mol_057: [('3V3M', 42), ('4YLU', 42)]
    pool_mol_074: [('3V3M', 51), ('4YLU', 51)]
    pool_mol_279: [('3V3M', 54), ('4YLU', 54)]


In [10]:
# === 枚举待算（有 md_fit.xtc、尚无 analysis.json）===
# Num 4
def list_targets():
    todo, done = [], []
    for arm in ARMS:
        for fit in sorted(glob.glob(f"{ROOT}/{arm}/*/*/06_md/{TRAJ}")):
            d = Path(fit).parent
            if "_validate" in str(d): continue
            res = OUT_ROOT / d.parents[1].name / d.parents[0].name / "analysis.json"
            (done if res.exists() else todo).append(d)
    return todo, done

todo, done = list_targets()
print(f"待算: {len(todo)}   已完成: {len(done)}  (md_fit 出齐后待算应 = {N_EXPECT})")

待算: 0   已完成: 36  (md_fit 出齐后待算应 = 36)


In [5]:
# === 写 per-轨迹 MDAnalysis 脚本（配体残基名由参数传入）===
# Num 5
analysis_py = r"""#!/usr/bin/env python3
import sys, json, warnings; warnings.filterwarnings('ignore')
import numpy as np
import MDAnalysis as mda
from MDAnalysis.analysis import rms, distances
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis

SRC, OUT = sys.argv[1], sys.argv[2]
LIG = sys.argv[3] if len(sys.argv) > 3 else 'MOL'      # V2=UNL / V3=MOL，由调用方传入
STRIDE = 10

u = mda.Universe(SRC + '/md.tpr', SRC + '/md_fit.xtc')
n_total = len(u.trajectory)
start = n_total // 2                       # 后 50 ns（10001 帧 -> 5000）
protein = u.select_atoms('protein')
ligand  = u.select_atoms(f'resname {LIG}')
assert len(ligand) > 0, f'ligand {LIG} not found'

R = rms.RMSD(u, reference=u, select='backbone',
             groupselections=[f'resname {LIG}'], ref_frame=0)
R.run(start=start, step=STRIDE)
rmsd_prot = float(np.mean(R.results.rmsd[:, 2]))
rmsd_lig  = float(np.mean(R.results.rmsd[:, 3]))

mindist_vals = []
for ts in u.trajectory[start::STRIDE]:
    d = distances.distance_array(ligand.positions, protein.positions)
    mindist_vals.append(float(d.min()) / 10.0)
mindist_mean = float(np.mean(mindist_vals)); n_frames = len(mindist_vals)

hb = HydrogenBondAnalysis(universe=u, donors_sel=f'resname {LIG} or protein',
                          acceptors_sel=f'resname {LIG} or protein',
                          d_a_cutoff=3.5, d_h_a_angle_cutoff=150.0,
                          update_selections=False)
hb.run(start=start, step=STRIDE)
frames_with_hb = 0
if hb.results.hbonds is not None and len(hb.results.hbonds) > 0:
    li, pr = set(ligand.indices), set(protein.indices)
    a = hb.results.hbonds
    d_l = np.isin(a[:, 1].astype(int), list(li)); a_p = np.isin(a[:, 3].astype(int), list(pr))
    d_p = np.isin(a[:, 1].astype(int), list(pr)); a_l = np.isin(a[:, 3].astype(int), list(li))
    frames_with_hb = len(np.unique(a[(d_l & a_p) | (d_p & a_l), 0]))
hbond_occ = frames_with_hb / n_frames if n_frames else 0.0

json.dump(dict(rmsd_prot_mean=round(rmsd_prot, 4), rmsd_lig_mean=round(rmsd_lig, 4),
               mindist_mean=round(mindist_mean, 4), hbond_occupancy=round(hbond_occ, 4),
               n_frames=n_frames, start_frame=start, n_total=n_total,
               ligand_resname=LIG, n_ligand_atoms=len(ligand)),
          open(OUT, 'w'), indent=1)
print(f'OK {LIG} n={len(ligand)} rmsd_prot={rmsd_prot:.3f} rmsd_lig={rmsd_lig:.3f} '
      f'mindist={mindist_mean:.3f} hbocc={hbond_occ:.3f}')
"""
PY_FILE = ROOT / "phase4_cross_v3_analysis_one.py"; PY_FILE.write_text(analysis_py)
print("已写 per-轨迹脚本:", PY_FILE.name)

已写 per-轨迹脚本: phase4_cross_v3_analysis_one.py


In [6]:
# === 生成 SLURM 数组 ===
# Num 6
LOGDIR = ROOT / "logs" / "phase4_cross_v3"; LOGDIR.mkdir(parents=True, exist_ok=True)
MANI = ROOT / "_phase4_cross_v3_manifest.txt"
MANI.write_text("\n".join(str(d) for d in todo) + "\n")

runner = f"""#!/usr/bin/env bash
set -uo pipefail
SRC="$1"
conda activate md_env 2>/dev/null || source activate md_env 2>/dev/null || true
MOL=$(basename "$(dirname "$(dirname "$SRC")")")
REP=$(basename "$(dirname "$SRC")")
WORK="{OUT_ROOT}/$MOL/$REP"; mkdir -p "$WORK"
[ -f "$WORK/analysis.json" ] && {{ echo "SKIP $MOL/$REP"; exit 0; }}
python3 {PY_FILE} "$SRC" "$WORK/analysis.json" "{LIG}"
"""
RUNNER = ROOT / "phase4_cross_v3_analysis_one.sh"; RUNNER.write_text(runner); os.chmod(RUNNER, 0o755)

gres = f"#SBATCH --gres={SBATCH['gres']}\n" if SBATCH["gres"] else ""
sb = f"""#!/usr/bin/env bash
#SBATCH --job-name=phase4_cross_v3_anal
#SBATCH --output={LOGDIR}/anal_%A_%a.out
#SBATCH --error={LOGDIR}/anal_%A_%a.err
#SBATCH --array=1-{max(len(todo),1)}%{CONCURRENCY}
#SBATCH --partition={SBATCH['partition']}
#SBATCH --account={SBATCH['account']}
#SBATCH --qos={SBATCH['qos']}
#SBATCH --cpus-per-task={SBATCH['cpus']}
#SBATCH --time={SBATCH['time']}
{gres}d=$(sed -n "${{SLURM_ARRAY_TASK_ID}}p" {MANI})
[ -z "$d" ] && exit 0
bash {RUNNER} "$d"
"""
SB = ROOT / "phase4_cross_v3_analysis.sbatch"; SB.write_text(sb)
print("已写:", MANI.name, RUNNER.name, SB.name)
print("日志:", LOGDIR)
print(f"提交: sbatch {SB}  (1-{len(todo)}%{CONCURRENCY})  配体 = {LIG}")
print("  （上为绝对路径，在任何工作目录下提交都有效）")

已写: _phase4_cross_v3_manifest.txt phase4_cross_v3_analysis_one.sh phase4_cross_v3_analysis.sbatch
日志: /mnt/hpc/home/wanghongmei/lang2/logs/phase4_cross_v3
提交: sbatch /mnt/hpc/home/wanghongmei/lang2/phase4_cross_v3_analysis.sbatch  (1-6%20)  配体 = MOL
  （上为绝对路径，在任何工作目录下提交都有效）


In [11]:
# === 收集 + 聚合（SLURM 完成后跑）===
# Num 7
rows = []
for jf in glob.glob(f"{OUT_ROOT}/*/*/analysis.json"):
    d = Path(jf)
    try: r = json.load(open(jf))
    except Exception: continue
    r.update(mol_dir=d.parent.parent.name, rep=d.parent.name); rows.append(r)
per = pd.DataFrame(rows)
print(f"解析到 {len(per)} 条  (目标 {N_EXPECT})")
if len(per):
    assert set(per.ligand_resname) == {LIG}, f"产物中残基名不一致: {set(per.ligand_resname)}"
    assert (per.n_ligand_atoms > 0).all(), "存在配体原子数为 0 的轨迹（选择串未命中）"
    print(per[["mol_dir", "rep", "rmsd_prot_mean", "rmsd_lig_mean",
               "mindist_mean", "hbond_occupancy"]].head(10).to_string(index=False))
    agg = per.groupby("mol_dir").agg(
        rmsd_prot_mean=("rmsd_prot_mean", "mean"),
        rmsd_lig_mean=("rmsd_lig_mean", "mean"),
        mindist_mean=("mindist_mean", "mean"),
        hbond_occupancy=("hbond_occupancy", "mean"),
        n_rep=("rep", "count")).round(4).reset_index()
    agg["mol_id"] = agg.mol_dir.str.split("__").str[0]
    agg["target"] = agg.mol_dir.str.split("__").str[1]
    out = ROOT / "results" / "phase4_cross_structural_SOLV_V3.csv"
    agg.to_csv(out, index=False)
    print("\n已存", out.name); print(agg.head(12).to_string(index=False))

解析到 36 条  (目标 36)
                    mol_dir rep  rmsd_prot_mean  rmsd_lig_mean  mindist_mean  hbond_occupancy
pool_mol_004__4YLU__primary  r3          1.9469         2.1069        0.1815           1.0000
pool_mol_004__4YLU__primary  r2          1.9844         3.9831        0.1924           1.0000
pool_mol_004__4YLU__primary  r1          1.7673         2.5125        0.1816           1.0000
pool_mol_012__3V3M__primary  r3          2.1298         2.9425        0.1919           0.5529
pool_mol_012__3V3M__primary  r2          2.5377         3.0391        0.1881           0.6088
pool_mol_012__3V3M__primary  r1          2.3973         2.7380        0.1910           0.5329
pool_mol_012__4YLU__primary  r3          2.3141         3.0370        0.1918           0.5469
pool_mol_012__4YLU__primary  r2          1.6273         7.7720        0.2143           0.2096
pool_mol_012__4YLU__primary  r1          1.3199         1.4965        0.1964           0.0419
pool_mol_057__4YLU__primary  r3          1

## 顺序
1. phase2 出齐 36 条 `md_fit.xtc` → 本 notebook「待算」= 36。（99 是自主臂复制而来的残留；在主臂 99 是对的，故只在跨臂改。）
2. 配体探测 cell 通过（含与 Node 05 I2b 的原子数对账）。
3. `sbatch phase4_cross_v3_analysis.sbatch` → `results/step6e_cross_v3/<mol>/<rep>/analysis.json`。
4. 收集 cell → `results/phase4_cross_structural_SOLV_V3.csv`。
5. 与 phase3 的 `phase3_dG_aggregated_SOLV_V3.csv` + BBS 合并 → phase5 COMPASS（六轴）。

## 与 V2 协议的一致性
- 后 50 ns、`STRIDE = 10`、`d_a = 3.5 Å` / `angle = 150°`、骨架叠合后算配体 RMSD —— 全同。
- 唯一实质改动是配体选择串由写死改为探测，且产物记录 `ligand_resname` / `n_ligand_atoms`
  以便日后无须再猜。